# Qwen2.5-7B-Instruct QLoRA Fine-Tuning for FitMyResume

Trains a LoRA adapter on the teacher-distilled SFT dataset (5,855 train / 726 validation rows) on a Colab Pro A100.

**Pipeline overview:**
1. Mount Drive, install dependencies
2. Load Qwen2.5-7B-Instruct in 4-bit quantization
3. Apply LoRA adapter
4. Load and format SFT data using Qwen's chat template
5. Train for 1 epoch
6. Save the adapter to Drive
7. Sanity-check inference on one example


## 0. Sanity check the runtime

Confirm you're actually on an A100 before installing anything. If this prints T4 or L4, change runtime and re-run.

In [ ]:
!nvidia-smi

Fri May 29 15:19:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P0             47W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. Mount Drive

We'll read SFT files from Drive and write the trained adapter back to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
DRIVE_BASE = '/content/drive/MyDrive/fit-my-resume'
SFT_TRAIN_PATH = f'{DRIVE_BASE}/data/instruction_tuning_train.jsonl'
SFT_VAL_PATH = f'{DRIVE_BASE}/data/instruction_tuning_validation.jsonl'
OUTPUT_DIR = f'{DRIVE_BASE}/models/qwen25-7b-fitmyresume-lora'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify the SFT files are reachable
assert os.path.exists(SFT_TRAIN_PATH), f'Missing: {SFT_TRAIN_PATH}'
assert os.path.exists(SFT_VAL_PATH), f'Missing: {SFT_VAL_PATH}'
print('SFT files found.')
print(f'  train: {SFT_TRAIN_PATH}')
print(f'  val:   {SFT_VAL_PATH}')
print(f'  output: {OUTPUT_DIR}')

SFT files found.
  train: /content/drive/MyDrive/fit-my-resume/data/instruction_tuning_train.jsonl
  val:   /content/drive/MyDrive/fit-my-resume/data/instruction_tuning_validation.jsonl
  output: /content/drive/MyDrive/fit-my-resume/models/qwen25-7b-fitmyresume-lora


## 2. Install dependencies

Pinning versions. Newer versions of `transformers`/`trl` sometimes break the SFTTrainer interface; these are known-good as of mid-2026.

In [ ]:
!pip install -q -U \
    'bitsandbytes>=0.45.0' \
    'transformers>=4.46.0' \
    'peft>=0.14.0' \
    'trl>=0.12.0' \
    'accelerate>=1.1.0' \
    'datasets>=3.1.0' \
    'sentencepiece' \
    'protobuf'

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('bf16 supported:', torch.cuda.is_bf16_supported())

CUDA available: True
Device: NVIDIA A100-SXM4-40GB
bf16 supported: True


## 3. Load Qwen2.5-7B-Instruct in 4-bit quantization

Qwen2.5-7B is Apache 2.0 —  The 4-bit quantization shrinks the model from ~14 GB to ~4 GB of VRAM, leaving plenty of room for LoRA gradients and the optimizer state.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False  # Required for gradient checkpointing during training
model.config.pretraining_tp = 1

print(f'Model loaded. Vocab size: {len(tokenizer)}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Model loaded. Vocab size: 151665


## 4. Apply LoRA adapter

Only LoRA parameters will be trained. The base model stays frozen and quantized.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


## 5. Load and format the dataset

Each SFT row has `instruction`, `input`, `output`, and `metadata` fields. We need to format these using Qwen's chat template so the model sees the same `<|im_start|>` / `<|im_end|>` structure it was trained on.

In [ ]:
from datasets import load_dataset

raw_train = load_dataset('json', data_files=SFT_TRAIN_PATH, split='train')
raw_val = load_dataset('json', data_files=SFT_VAL_PATH, split='train')

print(f'train rows: {len(raw_train)}')
print(f'val rows:   {len(raw_val)}')
print('\nSample row keys:', list(raw_train[0].keys()))
print('\nFirst instruction (first 200 chars):')
print(raw_train[0]['instruction'][:200])

train rows: 5855
val rows:   726

Sample row keys: ['instruction', 'input', 'output', 'metadata']

First instruction (first 200 chars):
Evaluate the resume against the job description. Return only valid JSON with score, explanation, and resume_suggestions. Do not invent experience.


In [ ]:
def format_for_qwen(example):
    """
    Build a single training string using Qwen's chat template.
    The model learns to predict the assistant turn given the system + user turns.
    """
    messages = [
        {'role': 'system', 'content': example['instruction']},
        {'role': 'user', 'content': example['input']},
        {'role': 'assistant', 'content': example['output']},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {'text': text}

train_ds = raw_train.map(format_for_qwen, remove_columns=raw_train.column_names)
val_ds = raw_val.map(format_for_qwen, remove_columns=raw_val.column_names)

print('Formatted train example (first 1000 chars):')
print(train_ds[0]['text'][:1000])
print('\n...')
print(train_ds[0]['text'][-500:])

Formatted train example (first 1000 chars):
<|im_start|>system
Evaluate the resume against the job description. Return only valid JSON with score, explanation, and resume_suggestions. Do not invent experience.<|im_end|>
<|im_start|>user
RESUME:
CONSULTANT Summary I am an experienced Program Manager, delivering enterprise-grade on-premises and SaaS products at Microsoft while being customer obsessed. I was previously an Enterprise Desktop Architect at multiple large companies, both as an employee and in a consulting capacity. I have a proven track record of positive impact in enterprise desktop management, infrastructure, systems administration, programming and automation, enterprise architecture, and project management. Highlights Windows OS VMware Server/View IIS Leadership System Center Configuration Manager Enterprise Imaging/OSD/MDT App-V Consulting MSI/Windows Installer BitLocker Full Disk Encryption Server 2K8/2k12 Project management InstallShield AdminStudio VDI ASP.NET/VB.NET/C

In [ ]:
# Sanity check: how long are the formatted sequences?
lengths = [len(tokenizer.encode(ex['text'])) for ex in train_ds.select(range(100))]
import statistics
print(f'Sample of 100 train rows:')
print(f'  min tokens:    {min(lengths)}')
print(f'  max tokens:    {max(lengths)}')
print(f'  mean tokens:   {statistics.mean(lengths):.0f}')
print(f'  median tokens: {statistics.median(lengths):.0f}')
print(f'  pct > 4096:    {sum(1 for x in lengths if x > 4096)}/100')

Sample of 100 train rows:
  min tokens:    1034
  max tokens:    5197
  mean tokens:   2401
  median tokens: 2308
  pct > 4096:    4/100


## 6. Configure SFTTrainer

Hyperparameters chosen for QLoRA on a 7B model with ~6k examples.

In [ ]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,  # effective batch size = 16
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    weight_decay=0.0,
    optim='paged_adamw_8bit',
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    max_length=4096,
    packing=False,  # Don't pack multiple examples per sequence; keeps loss honest
    dataset_text_field='text',
    logging_steps=10,
    save_strategy='steps',
    save_steps=500,
    save_total_limit=3,
    eval_strategy='steps',
    eval_steps=200,
    report_to='none',  # set to 'wandb' if you want experiment tracking
    seed=42,
    dataloader_num_workers=2,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=sft_config,
)

# Estimate total steps
total_steps = len(train_ds) // (sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps)
print(f'Estimated total training steps: {total_steps}')
print(f'Estimated training time on A100: ~{total_steps * 8 / 60:.0f} minutes (rough)')

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Estimated total training steps: 365
Estimated training time on A100: ~49 minutes (rough)


## 8. Full training

This is the long one. Expect ~2.5 hours. Loss should drop from ~2.0 at the start to ~0.4-0.7 by the end. Validation loss should track training loss closely; if it diverges (training keeps falling but val rises), that's overfitting.

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
200,1.825743,2.060789,1.937977,7260083.000000,0.569020
366,1.743081,2.066501,1.923170,13360095.000000,0.569573


TrainOutput(global_step=366, training_loss=1.871679824558112, metrics={'train_runtime': 8247.0911, 'train_samples_per_second': 0.71, 'train_steps_per_second': 0.044, 'total_flos': 6.582383681667871e+17, 'train_loss': 1.871679824558112, 'epoch': 1.0})

## 9. Save the final adapter

We save the LoRA adapter (small, ~150 MB) and the tokenizer. The base Qwen weights stay on Hugging Face

In [ ]:
FINAL_ADAPTER_PATH = f'{OUTPUT_DIR}/final'
trainer.model.save_pretrained(FINAL_ADAPTER_PATH)
tokenizer.save_pretrained(FINAL_ADAPTER_PATH)
print(f'Adapter saved to: {FINAL_ADAPTER_PATH}')
!ls -lh "{FINAL_ADAPTER_PATH}"

Adapter saved to: /content/drive/MyDrive/fit-my-resume/models/qwen25-7b-fitmyresume-lora/final
total 88M
-rw------- 1 root root 1.1K May 29 17:37 adapter_config.json
-rw------- 1 root root  78M May 29 17:37 adapter_model.safetensors
-rw------- 1 root root 2.5K May 29 17:37 chat_template.jinja
-rw------- 1 root root 5.1K May 29 17:37 README.md
-rw------- 1 root root  694 May 29 17:37 tokenizer_config.json
-rw------- 1 root root  11M May 29 17:37 tokenizer.json


## 10. Quick inference sanity check

Load the adapter on top of the base model and run inference on one validation example. Compare the model's output to the teacher's output — they should be in the same JSON shape, with similar score and reasoning.

In [ ]:
import json

# Pull one validation example
sample = raw_val[0]
print('=== INSTRUCTION ===')
print(sample['instruction'])
print('\n=== INPUT (first 500 chars) ===')
print(sample['input'][:500])
print('\n=== TEACHER OUTPUT (first 500 chars) ===')
print(sample['output'][:500])

=== INSTRUCTION ===
Evaluate the resume against the job description. Return only valid JSON with score, explanation, and resume_suggestions. Do not invent experience.

=== INPUT (first 500 chars) ===
RESUME:
NEW CONSTRUCTION / AFE MAINTENANCE MANAGER Summary 23 years in the Industry, with 18 years of Management / Supervisory experience in Shutdowns, T/A's, and Project related work. Safety statistics for all projects are always at the highest level. Proven track record as Manager / Supervisor managing a variety of construction teams and projects with a <0.5 trir.="" my="" experience is="" within all="" complexes of="" the="" benicia="" refinery.="" extensive="" background="" in="" many="" are

=== TEACHER OUTPUT (first 500 chars) ===
{"score":35,"explanation":{"matched_qualifications":["Managed construction schedules, manpower loading, and resource loading for refinery projects (e.g., VIP Project with 500 construction employees).","Oversaw project budgets and cost control, including bud

In [ ]:
# Build the prompt the same way as training, but stop before the assistant turn
messages = [
    {'role': 'system', 'content': sample['instruction']},
    {'role': 'user', 'content': sample['input']},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

# Enable cache for inference
model.config.use_cache = True
model.eval()

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=False,  # deterministic for evaluation
        temperature=1.0,
        pad_token_id=tokenizer.pad_token_id,
    )

generated = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('=== STUDENT OUTPUT ===')
print(generated)

# Try to parse as JSON to confirm schema fidelity
try:
    parsed = json.loads(generated)
    print('\n✓ Parsed as valid JSON')
    print(f'  score: {parsed.get("score")}')
    print(f'  suggestions: {len(parsed.get("resume_suggestions", []))}')
except json.JSONDecodeError as e:
    print(f'\n✗ JSON parse failed: {e}')

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


=== STUDENT OUTPUT ===
{"score":45,"explanation":{"matched_qualifications":["Over 18 years of construction management experience, including shutdowns, turnaround (T/A), and project-related work.","Proven track record of managing large-scale projects with budgets, schedules, and headcounts (e.g., 500+ construction employees).","Experience in safety oversight, site safety plan implementation, and environmental compliance.","Familiarity with contractor coordination, bidding, and contract management.","Demonstrated ability to manage multiple projects simultaneously and ensure on-time completion."],"missing_or_weak_qualifications":["No explicit experience with construction project management on the owner or general contractor side; resume focuses on refinery operations and maintenance.","No mention of managing GC vendor invoices, tracking against project budget, or forecasting spend.","No evidence of managing the transition from construction to operations through training and turnover.","No